In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# === Load dataset ===
df = pd.read_csv("bar_pass_prediction (processed version).csv")

# === Define label and features ===
X = df.drop(columns='bar_passed')
y = df['bar_passed']

# === Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# === Convert object/bool columns to numeric ===
for df_ in [X_train, X_test]:
    for col in df_.select_dtypes(include=['object', 'bool']).columns:
        unique_vals = df_[col].dropna().unique().tolist()
        if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
            df_[col] = df_[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
        else:
            df_[col] = LabelEncoder().fit_transform(df_[col].astype(str))

# === Define privileged/unprivileged groups based on race ===
group_priv = X_test['race'] == 7
group_unpriv = X_test['race'] != 7

# === Define models ===
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

# === Print header ===
print("\nFairness Metrics (Privileged: race == 7)")
print("--------------------------------------------------------------------------")
print(f"{'Model':<20} {'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")

# === Evaluate each model ===
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # === ERD ===
    err_priv = np.mean(y_pred[group_priv] != y_test[group_priv])
    err_unpriv = np.mean(y_pred[group_unpriv] != y_test[group_unpriv])
    erd = err_unpriv - err_priv

    # === TPRD ===
    def tpr(y_true, y_pred):
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        return tp / (tp + fn) if (tp + fn) > 0 else 0

    tpr_priv = tpr(y_test[group_priv], y_pred[group_priv])
    tpr_unpriv = tpr(y_test[group_unpriv], y_pred[group_unpriv])
    tprd = tpr_unpriv - tpr_priv

    # === ABROCA ===
    def compute_roc_auc(y_true, y_score):
        fpr, tpr_vals, _ = roc_curve(y_true, y_score)
        return auc(fpr, tpr_vals)

    abroca = np.nan
    if y_prob is not None:
        auc_priv = compute_roc_auc(y_test[group_priv], y_prob[group_priv])
        auc_unpriv = compute_roc_auc(y_test[group_unpriv], y_prob[group_unpriv])
        abroca = abs(auc_priv - auc_unpriv)

    # === Fairness score ===
    if not np.isnan(abroca):
        fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3
    else:
        fairness = float("nan")

    # === Print result ===
    print(f"{name:<20} {abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (Privileged: race == 7)
--------------------------------------------------------------------------
Model                ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------
Decision Tree        0.0774     0.1350     -0.1053    0.8941    
Logistic Regression  0.0191     0.0880     -0.0284    0.9548    
Random Forest        0.0856     0.0937     -0.0368    0.9280    
SVM                  0.0680     0.0895     0.0000     0.9475    
XGBoost              0.0444     0.0981     -0.0382    0.9397    


In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# === Load dataset ===
df = pd.read_csv("synthetic_law_data_decaf.csv")

# === Define label and features ===
X = df.drop(columns='bar_passed')
y = df['bar_passed']

# === Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# === Convert object/bool columns to numeric ===
for df_ in [X_train, X_test]:
    for col in df_.select_dtypes(include=['object', 'bool']).columns:
        unique_vals = df_[col].dropna().unique().tolist()
        if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
            df_[col] = df_[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
        else:
            df_[col] = LabelEncoder().fit_transform(df_[col].astype(str))

# === Define privileged/unprivileged groups based on race ===
group_priv = X_test['race'] == 7
group_unpriv = X_test['race'] != 7

# === Define models ===
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

# === Print header ===
print("\nFairness Metrics (Privileged: race == 7)")
print("--------------------------------------------------------------------------")
print(f"{'Model':<20} {'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")

# === Evaluate each model ===
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # === ERD ===
    err_priv = np.mean(y_pred[group_priv] != y_test[group_priv])
    err_unpriv = np.mean(y_pred[group_unpriv] != y_test[group_unpriv])
    erd = err_unpriv - err_priv

    # === TPRD ===
    def tpr(y_true, y_pred):
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        return tp / (tp + fn) if (tp + fn) > 0 else 0

    tpr_priv = tpr(y_test[group_priv], y_pred[group_priv])
    tpr_unpriv = tpr(y_test[group_unpriv], y_pred[group_unpriv])
    tprd = tpr_unpriv - tpr_priv

    # === ABROCA ===
    def compute_roc_auc(y_true, y_score):
        fpr, tpr_vals, _ = roc_curve(y_true, y_score)
        return auc(fpr, tpr_vals)

    abroca = np.nan
    if y_prob is not None:
        auc_priv = compute_roc_auc(y_test[group_priv], y_prob[group_priv])
        auc_unpriv = compute_roc_auc(y_test[group_unpriv], y_prob[group_unpriv])
        abroca = abs(auc_priv - auc_unpriv)

    # === Fairness score ===
    if not np.isnan(abroca):
        fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3
    else:
        fairness = float("nan")

    # === Print result ===
    print(f"{name:<20} {abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (Privileged: race == 7)
--------------------------------------------------------------------------
Model                ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------
Decision Tree        nan        -0.0757    0.0447     nan       
Logistic Regression  nan        -0.0324    0.0000     nan       


c:\Users\sshakibhamedan\Anaconda3\lib\site-packages\sklearn\metrics\_ranking.py:1137: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(
c:\Users\sshakibhamedan\Anaconda3\lib\site-packages\sklearn\metrics\_ranking.py:1137: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(


Random Forest        nan        -0.0324    0.0000     nan       
SVM                  nan        -0.0324    0.0000     nan       
XGBoost              nan        -0.0324    0.0000     nan       


c:\Users\sshakibhamedan\Anaconda3\lib\site-packages\sklearn\metrics\_ranking.py:1137: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(
c:\Users\sshakibhamedan\Anaconda3\lib\site-packages\sklearn\metrics\_ranking.py:1137: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(
c:\Users\sshakibhamedan\Anaconda3\lib\site-packages\sklearn\metrics\_ranking.py:1137: UndefinedMetricWarning: No negative samples in y_true, false positive value should be meaningless
  warnings.warn(


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# === Load dataset ===
df = pd.read_csv("generated_data_CLLM_prompt_Law.csv")

# === Define label and features ===
X = df.drop(columns='bar_passed')
y = df['bar_passed']

# === Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# === Convert object/bool columns to numeric ===
for df_ in [X_train, X_test]:
    for col in df_.select_dtypes(include=['object', 'bool']).columns:
        unique_vals = df_[col].dropna().unique().tolist()
        if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
            df_[col] = df_[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
        else:
            df_[col] = LabelEncoder().fit_transform(df_[col].astype(str))

# === Define privileged/unprivileged groups based on race ===
group_priv = X_test['race'] == 7
group_unpriv = X_test['race'] != 7

# === Define models ===
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

# === Print header ===
print("\nFairness Metrics (Privileged: race == 7)")
print("--------------------------------------------------------------------------")
print(f"{'Model':<20} {'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")

# === Evaluate each model ===
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # === ERD ===
    err_priv = np.mean(y_pred[group_priv] != y_test[group_priv])
    err_unpriv = np.mean(y_pred[group_unpriv] != y_test[group_unpriv])
    erd = err_unpriv - err_priv

    # === TPRD ===
    def tpr(y_true, y_pred):
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        return tp / (tp + fn) if (tp + fn) > 0 else 0

    tpr_priv = tpr(y_test[group_priv], y_pred[group_priv])
    tpr_unpriv = tpr(y_test[group_unpriv], y_pred[group_unpriv])
    tprd = tpr_unpriv - tpr_priv

    # === ABROCA ===
    def compute_roc_auc(y_true, y_score):
        fpr, tpr_vals, _ = roc_curve(y_true, y_score)
        return auc(fpr, tpr_vals)

    abroca = np.nan
    if y_prob is not None:
        auc_priv = compute_roc_auc(y_test[group_priv], y_prob[group_priv])
        auc_unpriv = compute_roc_auc(y_test[group_unpriv], y_prob[group_unpriv])
        abroca = abs(auc_priv - auc_unpriv)

    # === Fairness score ===
    if not np.isnan(abroca):
        fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3
    else:
        fairness = float("nan")

    # === Print result ===
    print(f"{name:<20} {abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (Privileged: race == 7)
--------------------------------------------------------------------------
Model                ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------
Decision Tree        0.0951     0.0977     -0.1277    0.8932    
Logistic Regression  0.0109     0.0575     -0.0957    0.9453    
Random Forest        0.0215     0.0805     -0.1277    0.9234    
SVM                  0.0311     0.0093     -0.0137    0.9820    
XGBoost              0.0215     0.0862     -0.1064    0.9286    


In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# === Load dataset ===
df = pd.read_csv("generated_data_Our_prompts_Law.csv")

# === Define label and features ===
X = df.drop(columns='bar_passed')
y = df['bar_passed']

# === Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2, random_state=42)

# === Convert object/bool columns to numeric ===
for df_ in [X_train, X_test]:
    for col in df_.select_dtypes(include=['object', 'bool']).columns:
        unique_vals = df_[col].dropna().unique().tolist()
        if set(unique_vals).issubset({'True', 'False', 'true', 'false'}):
            df_[col] = df_[col].astype(str).map({'True': 1, 'False': 0, 'true': 1, 'false': 0})
        else:
            df_[col] = LabelEncoder().fit_transform(df_[col].astype(str))

# === Define privileged/unprivileged groups based on race ===
group_priv = X_test['race'] == 7
group_unpriv = X_test['race'] != 7

# === Define models ===
models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'XGBoost': XGBClassifier(eval_metric='logloss', random_state=42)
}

# === Print header ===
print("\nFairness Metrics (Privileged: race == 7)")
print("--------------------------------------------------------------------------")
print(f"{'Model':<20} {'ABROCA':<10} {'ERD':<10} {'TPRD':<10} {'Fairness':<10}")
print("--------------------------------------------------------------------------")

# === Evaluate each model ===
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # === ERD ===
    err_priv = np.mean(y_pred[group_priv] != y_test[group_priv])
    err_unpriv = np.mean(y_pred[group_unpriv] != y_test[group_unpriv])
    erd = err_unpriv - err_priv

    # === TPRD ===
    def tpr(y_true, y_pred):
        tp = np.sum((y_true == 1) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))
        return tp / (tp + fn) if (tp + fn) > 0 else 0

    tpr_priv = tpr(y_test[group_priv], y_pred[group_priv])
    tpr_unpriv = tpr(y_test[group_unpriv], y_pred[group_unpriv])
    tprd = tpr_unpriv - tpr_priv

    # === ABROCA ===
    def compute_roc_auc(y_true, y_score):
        fpr, tpr_vals, _ = roc_curve(y_true, y_score)
        return auc(fpr, tpr_vals)

    abroca = np.nan
    if y_prob is not None:
        auc_priv = compute_roc_auc(y_test[group_priv], y_prob[group_priv])
        auc_unpriv = compute_roc_auc(y_test[group_unpriv], y_prob[group_unpriv])
        abroca = abs(auc_priv - auc_unpriv)

    # === Fairness score ===
    if not np.isnan(abroca):
        fairness = (3 - abs(abroca) - abs(erd) - abs(tprd)) / 3
    else:
        fairness = float("nan")

    # === Print result ===
    print(f"{name:<20} {abroca:<10.4f} {erd:<10.4f} {tprd:<10.4f} {fairness:<10.4f}")



Fairness Metrics (Privileged: race == 7)
--------------------------------------------------------------------------
Model                ABROCA     ERD        TPRD       Fairness  
--------------------------------------------------------------------------
Decision Tree        0.0247     0.0230     -0.0265    0.9753    
Logistic Regression  0.0077     -0.0013    0.0582     0.9776    
Random Forest        0.0011     0.0270     -0.0053    0.9889    
SVM                  0.0082     -0.0357    0.0899     0.9554    
XGBoost              0.0017     -0.0102    -0.0212    0.9890    
